# Load images from folders and feed them to a PyTorch model

This notebook shows how to:
1. Download image data to the local disk.
2. Save each image into class folders.
3. Load those images directly with `ImageFolder`.
4. Train a small image classifier.

In [ ]:
# Install packages if needed
# %pip install torch torchvision pillow

In [1]:
import os
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10, ImageFolder
from PIL import Image

In [2]:
# Download CIFAR10 images to disk as PNG files

root_dir = Path('cifar10_images')
download_root = Path('cifar10_download')

classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

for cls in classes:
    (root_dir / cls).mkdir(parents=True, exist_ok=True)

train_ds = CIFAR10(root=str(download_root), download=True, train=True)

for idx, (img, label) in enumerate(train_ds):
    cls_name = classes[label]
    save_path = root_dir / cls_name / f'{idx}.png'
    img.save(save_path)

print(f'Created images in {root_dir}')

100%|██████████| 170M/170M [55:12<00:00, 51.5kB/s]   


Created images in cifar10_images


In [3]:
# Load the saved image files directly

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = ImageFolder(root=str(root_dir), transform=transform)
print(dataset.classes)
print(dataset[0][0].shape)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
torch.Size([3, 32, 32])


In [4]:
# Create loaders

train_loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

print('Number of batches:', len(train_loader))

Number of batches: 782


In [5]:
# A simple CNN

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [6]:
# Train for a few epochs

for epoch in range(2):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch + 1}, loss: {running_loss / len(train_loader):.4f}')

Epoch 1, loss: 1.4507
Epoch 2, loss: 1.1061


## If you already have your own image folders

You can skip the download step and point `ImageFolder` directly to a folder like this:

```python
dataset = ImageFolder(root='path/to/your/images', transform=transform)
```

Each subfolder becomes one class. For example:

```text
path/to/your/images/
  cats/
    1.jpg
    2.jpg
  dogs/
    1.jpg
```